# Ensemble stability

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
ens_attr_paths = {
    '10m_3L': {
        'attrs_paths': [
            "./experiments/ensemble_10m_3L/e0/attributions.csv",
            "./experiments/ensemble_10m_3L/e1/attributions.csv",
            "./experiments/ensemble_10m_3L/e2/attributions.csv",
            "./experiments/ensemble_10m_3L/e3/attributions.csv",
            "./experiments/ensemble_10m_3L/e4/attributions.csv",
            "./experiments/ensemble_10m_3L/e5/attributions.csv",
            "./experiments/ensemble_10m_3L/e6/attributions.csv",
            "./experiments/ensemble_10m_3L/e7/attributions.csv",
            "./experiments/ensemble_10m_3L/e8/attributions.csv",
            "./experiments/ensemble_10m_3L/e9/attributions.csv",
        ],
        'n_layers': 3,
    },
    '5m_2L': {
        'attrs_paths': [
            "./experiments/ensemble_5m_2L/e0/attributions.csv",
            "./experiments/ensemble_5m_2L/e1/attributions.csv",
            "./experiments/ensemble_5m_2L/e2/attributions.csv",
            "./experiments/ensemble_5m_2L/e3/attributions.csv",
            "./experiments/ensemble_5m_2L/e4/attributions.csv",
        ],
        'n_layers': 2,
    }, 
    '5m_4L': {
        'attrs_paths': [
            "./experiments/ensemble_5m_4L/e0/attributions.csv",
            "./experiments/ensemble_5m_4L/e1/attributions.csv",
            "./experiments/ensemble_5m_4L/e2/attributions.csv",
            "./experiments/ensemble_5m_4L/e3/attributions.csv",
            "./experiments/ensemble_5m_4L/e4/attributions.csv",
        ],
        'n_layers': 4,
    }
}

ens_key = '10m_3L'
attrs_paths = ens_attr_paths[ens_key]['attrs_paths']
n_layers = ens_attr_paths[ens_key]['n_layers']
n_models = len(attrs_paths)

In [ ]:
# list of attribution dataframes
# (each df with attrs for all algos per ensemble member)
scores_per_model = []
for path in attrs_paths:
    assert os.path.exists(path), f"File not found: {path}"
    df = pd.read_csv(path, index_col='snp')
    scores_per_model.append(df)

algos = scores_per_model[0].columns.tolist()

# dict of attribution dataframes
# (each df with attrs for one algo across all ensemble members)
algo_dfs = {algo: pd.DataFrame() for algo in algos}
for i, df in enumerate(scores_per_model):
    for algo in algos:
        algo_dfs[algo][f'e{i}'] = df[algo]

In [ ]:
algos_sg = [algo for algo in algos if algo.endswith("SG")]
algos_nosg = [algo for algo in algos if not algo.endswith("SG")]

In [ ]:
def rsd(df):
    # Relative standard deviation: std / mean 
    return df.std(axis=1) / (df.mean(axis=1) + 1e-12)

def summarize_dispersion(s):
    """
    Summarize dispersion across variability measure. 
    
    Median Absolute Deviation (MAD) is a measure of dispersion
    that is computed as the median of the absolute deviations 
    from the median: 
        MAD = median(|Xj - median(X)|)
    where Xj represents the individual STDs/RSDs for SNP j 
    across ensemble members and median(X) is the median STD/RSD
    across all SNPs (J).

    MAD is scaled to get a measure that is comparable to 
    standard deviation under normality:
        (MAD_scaled = 1.4826 * MAD)
    """
    mean, med = s.mean(), s.median()
    mad = (s - med).abs().median()
    mad_scaled = 1.4826 * mad           # comparable to SD if normal
    q1, q3 = s.quantile([0.25, 0.75])
    p10, p90 = s.quantile([0.10, 0.90])
    return pd.Series({
        "mean": mean, 
        "median": med,
        "mad": mad,
        "mad_scaled": mad_scaled,
        "q1": q1, "q3": q3, 
        "iqr": q3 - q1,
        "p10": p10, "p90": p90
    })

rsd_df = pd.DataFrame({
    algo: rsd(df) for algo, df in algo_dfs.items()
})

ens_rsd_summary_df = pd.DataFrame({
    algo: summarize_dispersion(rsd_df[algo]) 
    for algo in rsd_df.columns
}).T
ens_rsd_summary_df

In [ ]:
ens_summary_file = "./experiments/ensemble_10m_3L/ens_stability.csv"
ens_rsd_summary_df.to_csv(ens_summary_file, index_label='algorithm')
print(f"Ensemble stability results saved to:\n{ens_summary_file}")

In [ ]:
algos_sg = [algo for algo in algos if algo.endswith("SG")]
algos_nosg = [algo for algo in algos if not algo.endswith("SG")]
print(f"Number of ensemble members: {n_models}")
print(f"Total number of algorithms: {len(algos_sg) + len(algos_nosg)}")
print(f"Algorithms with SmoothGrad: {algos_sg}")
print(f"Algorithms without SmoothGrad: {algos_nosg}")

In [ ]:
melted_col_name = 'rsd'
melt_df = rsd_df.melt(var_name="algorithm", value_name=melted_col_name)
melt_df["algorithm_base"] = melt_df["algorithm"].str.replace(r"SG$", "", regex=True)
melt_df['variation'] = np.where(melt_df["algorithm"].str.endswith("SG"), 
                                "SmoothGrad", "No SmoothGrad")
plt.figure(figsize=(7.4, 4.8))
sns.violinplot(
    data=melt_df,
    x="algorithm_base", 
    y=melted_col_name,
    hue="variation",
    split=True,
    hue_order=["No SmoothGrad", 
               "SmoothGrad"],
    inner='box',
)
plt.xticks(rotation=10, ha="center")
plt.ylabel("relative standard deviation")
plt.title(f"Ensemble Consistency")
plt.grid(alpha=0.5)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.xlabel("")
plt.tight_layout()
save_path = f"./ens_stability_violinplot.tiff"
plt.savefig(
    save_path,
    dpi=300, 
    bbox_inches='tight',
    format='tiff',
    pil_kwargs={"compression": "tiff_lzw"}
)
print(f"Plot saved to: {save_path}")
plt.show()